# Análisis exploratorio y preparación de datos — RetailNova

Este notebook explora la cohorte sintética de RetailNova y documenta las decisiones de preparación previas al modelado. Su objetivo es orientar el diseño de la pipeline; no sustituye la validación con datos comerciales reales.

## 1. Fuente, objetivos y diccionario de variables

**Qué haremos:** describiremos la fuente sintética con cobertura histórica mensual de 36 meses, el objetivo analítico y el significado de las variables principales.

**Por qué es relevante:** conocer el corte temporal y la definición de abandono permite distinguir predictores disponibles de información futura que causaría fuga de datos. El diccionario también hace comprensibles las variables para el equipo de negocio y para futuras decisiones de modelado.

In [1]:
from IPython.display import display
import pandas as pd

objetivos = pd.DataFrame([
    ('Objetivo de negocio', 'Explorar señales para priorizar acciones de fidelización de clientes.'),
    ('Objetivo analítico', 'Predecir churn_6m: cero compras entre 2026-04-01 y 2026-10-01.'),
    ('Corte de predictores', 'Comportamiento observado hasta antes del 2026-04-01.'),
    ('Alcance', 'Datos sintéticos; no representan clientes ni transacciones reales.'),
], columns=['aspecto', 'definicion'])
diccionario = pd.DataFrame([
    ('id_cliente', 'Customer ID', 'Identificador; trazabilidad, no predictor.'),
    ('edad', 'Age', 'Edad simulada; contiene faltantes.'),
    ('comuna', 'Commune / locality', 'Comuna simulada; uso descriptivo, revisar privacidad y sesgo.'),
    ('region', 'Region', 'Región simulada; uso descriptivo y auditoría.'),
    ('fecha_registro', 'Registration date', 'Fecha de alta; no se usa como fecha cruda del modelo.'),
    ('canal_compra', 'Purchase channel', 'Canal predominante: tienda, comercio electrónico o mixto.'),
    ('antiguedad_meses', 'Tenure in months', 'Antigüedad del cliente al corte.'),
    ('cantidad_compras_12m', 'Purchases in prior 12 months', 'Número de compras antes del corte.'),
    ('frecuencia_compra_mensual', 'Monthly purchase frequency', 'Compras en 12 meses divididas por 12; redundante con cantidad_compras_12m.'),
    ('monto_promedio_compra', 'Average purchase value', 'Valor promedio de compra en moneda simulada.'),
    ('monto_total_12m', 'Total spend in prior 12 months', 'Monto promedio por cantidad de compras; variable derivada.'),
    ('categoria_principal', 'Main product category', 'Categoría de compra predominante.'),
    ('proporcion_compras_promocion', 'Promotion purchase share', 'Proporción de compras con promoción.'),
    ('devoluciones_12m', 'Returns in prior 12 months', 'Cantidad de devoluciones antes del corte.'),
    ('reclamos_12m', 'Complaints in prior 12 months', 'Cantidad de reclamos antes del corte.'),
    ('interacciones_campanas_12m', 'Campaign interactions', 'Interacciones con campañas antes del corte.'),
    ('fecha_ultima_compra_pre_corte', 'Last purchase before cutoff', 'Fecha de compra previa; recencia ya resume la distancia temporal.'),
    ('recencia_dias', 'Purchase recency in days', 'Días desde la última compra previa al corte.'),
    ('compras_periodo_objetivo', 'Purchases in outcome window', 'Información futura usada para construir la etiqueta; nunca predictor.'),
    ('fecha_ultima_compra_al_cierre', 'Last purchase at outcome end', 'Información futura; nunca predictor.'),
    ('churn_6m', 'Six-month churn', 'Objetivo: 1 si no hubo compras durante la ventana posterior.'),
], columns=['variable', 'traduccion', 'definicion_y_uso'])
display(objetivos)
display(diccionario)

                aspecto                                         definicion
0   Objetivo de negocio  Explorar señales para priorizar acciones de fi...
1    Objetivo analítico  Predecir churn_6m: cero compras entre 2026-04-...
2  Corte de predictores  Comportamiento observado hasta antes del 2026-...
3               Alcance  Datos sintéticos; no representan clientes ni t...
                         variable  ...                                   definicion_y_uso
0                      id_cliente  ...         Identificador; trazabilidad, no predictor.
1                            edad  ...                 Edad simulada; contiene faltantes.
2                          comuna  ...  Comuna simulada; uso descriptivo, revisar priv...
3                          region  ...      Región simulada; uso descriptivo y auditoría.
4                  fecha_registro  ...  Fecha de alta; no se usa como fecha cruda del ...
5                    canal_compra  ...  Canal predominante: tienda, comercio electrón

### Conclusión de la sección 1

El objetivo se define en una ventana posterior al corte, mientras que los predictores deben describir únicamente los 12 meses previos. Los identificadores, las fechas posteriores y las compras de la ventana objetivo no son entradas del modelo. El diccionario traduce los campos al lenguaje de retail y deja explícitas variables derivadas o destinadas solo a auditoría.

## 2. Configuración del entorno y carga

**Qué haremos:** cargaremos el resumen sintético por cliente y comprobaremos la cobertura del historial mensual agregado de 36 meses.

**Por qué es relevante:** los datos del caso abarcan 36 meses, aunque los predictores se calcularán en una ventana de 12 meses antes del corte para estimar abandono en los seis meses posteriores. Verificar ambas ventanas hace explícita la temporalidad y facilita detectar fuga de información.

In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

ROOT = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / 'src' / 'paths.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('No se encontró src/paths.py en esta carpeta ni en sus directorios padre.')
sys.path.insert(0, str(ROOT / 'src'))
from paths import (
    DASHBOARD_TRAIN_PATH, DATASET_PATH, GRAPHS_DIR as GRAPH_DIR,
    HISTORICAL_MONTHLY_PATH, METADATA_PATH, MODEL_DIR, PROCESSED_DIR,
    PROCESSED_TEST_DIR, PROCESSED_TRAIN_DIR, RAW_DIR, RAW_TEST_DIR,
    RAW_TRAIN_DIR, TEST_RAW_PATH, TRAIN_CLEAN_PATH, TRAIN_RAW_PATH,
    TRAIN_SOURCE_PATH,
)
for directory in [
    RAW_DIR, PROCESSED_DIR, RAW_TRAIN_DIR, RAW_TEST_DIR,
    PROCESSED_TRAIN_DIR, PROCESSED_TEST_DIR, GRAPH_DIR, MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATASET_PATH
PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
sns.set_theme(style='whitegrid', context='notebook')

def save_figure(figure, filename):
    path = GRAPH_DIR / filename
    figure.savefig(path, dpi=220, bbox_inches='tight', facecolor='white')
    plt.close(figure)
    print(f'Gráfico guardado: {path.relative_to(ROOT)}')

def annotate_count_bars(axis, denominator):
    axis.margins(y=0.16)
    for patch in axis.patches:
        value = patch.get_height()
        if value > 0:
            axis.annotate(
                f'{value:,.0f}\n{value / denominator:.1%}',
                (patch.get_x() + patch.get_width() / 2, value),
                ha='center', va='bottom', fontsize=8
            )

df = pd.read_csv(DATA_PATH)
history_metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))['historial_mensual']
historical_monthly = pd.read_csv(HISTORICAL_MONTHLY_PATH, usecols=['mes'])
historical_monthly['mes'] = pd.to_datetime(historical_monthly['mes'])
assert historical_monthly['mes'].dt.to_period('M').nunique() == 36
print(f"Cobertura histórica: {historical_monthly['mes'].min():%Y-%m} a {historical_monthly['mes'].max():%Y-%m} ({historical_monthly['mes'].dt.to_period('M').nunique()} meses)")
print(f'Archivo: {DATA_PATH.relative_to(ROOT)}')
print(f'Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Cobertura histórica: 2023-04 a 2026-03 (36 meses)
Archivo: data\raw\dataset.csv
Dimensiones: 185,000 filas x 23 columnas


### Conclusión de la sección 2

La fuente se cargó desde `data/raw/dataset.csv` y dispone de 36 meses de historia agregada en `data/raw/historial_mensual_36m.csv.gz`. El modelo trabaja con variables calculadas sobre los 12 meses previos al corte; los splits y artefactos mantienen sus rutas centralizadas en `src/paths.py`.

## 3. Inspección inicial

**Qué haremos:** inspeccionaremos tipos, valores no nulos, memoria, muestra y esquema del dataset mediante `df.info()`.

**Por qué es relevante:** esta revisión permite detectar problemas de tipos y faltantes antes de transformar los datos. En especial, diferencia variables numéricas, categóricas y fechas, que requieren tratamientos distintos.

In [3]:
df.info()
display(df.head())
display(df.dtypes.rename('tipo').to_frame())

<class 'pandas.DataFrame'>
RangeIndex: 185000 entries, 0 to 184999
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   id_cliente                     185000 non-null  int64  
 1   edad                           170200 non-null  float64
 2   comuna                         177600 non-null  str    
 3   region                         185000 non-null  str    
 4   fecha_registro                 185000 non-null  str    
 5   canal_compra                   185000 non-null  str    
 6   antiguedad_meses               185000 non-null  int64  
 7   cantidad_compras_12m           185000 non-null  int64  
 8   frecuencia_compra_mensual      185000 non-null  float64
 9   monto_promedio_compra          185000 non-null  float64
 10  monto_total_12m                185000 non-null  float64
 11  categoria_principal            185000 non-null  str    
 12  proporcion_compras_promocion   185000 non

### Conclusión de la sección 3

`df.info()` permite verificar el esquema y los campos incompletos; las fechas se cargan como texto y deberán excluirse como crudas o convertirse si se derivan atributos. La inspección se realiza antes del split solo para conocer el esquema: no se calculan estadísticas de predictores ni se ajustan transformaciones sobre el conjunto completo.

## 4. Separación de entrenamiento y test (80/20)

**Qué haremos:** separaremos una única vez 80% para entrenamiento y 20% como holdout, estratificando por la etiqueta, y guardaremos test inmediatamente.

**Por qué es relevante:** el test debe permanecer fuera del análisis, selección de variables y ajuste de transformaciones. Se reservará para la revisión final del modelo. A partir de este punto las celdas exploratorias consultan exclusivamente `train_df`.

In [4]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=df['churn_6m']
)
train_df = df.loc[train_idx].copy().reset_index(drop=True)
test_holdout = df.loc[test_idx].copy().reset_index(drop=True)
raw_train_path = TRAIN_RAW_PATH
test_path = TEST_RAW_PATH
train_df.to_csv(raw_train_path, index=False)
test_holdout.to_csv(test_path, index=False)
print(f'Train raw guardado: {raw_train_path.relative_to(ROOT)}')
print(f'Train: {len(train_df):,} ({len(train_df) / len(df):.0%})')
print(f'Test reservado: {len(test_holdout):,} ({len(test_holdout) / len(df):.0%})')
print(f'Test guardado para revisión final: {test_path.relative_to(ROOT)}')
del test_holdout
for date_col in ['fecha_registro', 'fecha_ultima_compra_pre_corte', 'fecha_ultima_compra_al_cierre']:
    train_df[date_col] = pd.to_datetime(train_df[date_col], errors='raise')
del df

Train raw guardado: data\raw\train\train.csv
Train: 148,000 (80%)
Test reservado: 37,000 (20%)
Test guardado para revisión final: data\raw\test\test.csv


### Conclusión de la sección 4

Quedó reservado un 20% estratificado en `data/raw/test/test.csv`; el 80% restante se guarda en `data/raw/train/train.csv` y es la única fuente para el EDA y la selección. No volveremos a cargar ni consultar el holdout desde este notebook. La estratificación usa la etiqueta solo para preservar proporciones de clase, no para ajustar predictores.

## 5. Limpieza inicial de train

**Qué haremos:** detectaremos duplicados y faltantes en train, separaremos variables de trazabilidad y excluiremos campos futuros, fechas crudas e identificadores del conjunto de predictores.

**Por qué es relevante:** limpiar después del split evita que decisiones guiadas por el test influyan en el entrenamiento. Los faltantes se imputarán dentro de la pipeline, que ajustará sus reglas solo con los datos de entrenamiento de cada ajuste.

In [5]:
duplicate_count = int(train_df.duplicated().sum())
train_df = train_df.drop_duplicates().copy()
missing_report = train_df.isna().sum().to_frame('faltantes')
missing_report['porcentaje'] = (missing_report.faltantes / len(train_df) * 100).round(2)
display(missing_report.loc[missing_report.faltantes > 0])
print(f'Filas duplicadas eliminadas de train: {duplicate_count}')

TARGET = 'churn_6m'
ID = 'id_cliente'
LEAKAGE_AND_RAW_DATE = [
    'fecha_registro', 'fecha_ultima_compra_pre_corte',
    'compras_periodo_objetivo', 'fecha_ultima_compra_al_cierre'
]
EXCLUDE_FROM_MODEL = [ID, *LEAKAGE_AND_RAW_DATE, TARGET]
print('Excluidas de X:', EXCLUDE_FROM_MODEL)
print('La imputación se ajustará dentro del pipeline usando train, no sobre test.')

        faltantes  porcentaje
edad        11770        7.95
comuna       5900        3.99
Filas duplicadas eliminadas de train: 0
Excluidas de X: ['id_cliente', 'fecha_registro', 'fecha_ultima_compra_pre_corte', 'compras_periodo_objetivo', 'fecha_ultima_compra_al_cierre', 'churn_6m']
La imputación se ajustará dentro del pipeline usando train, no sobre test.


### Conclusión de la sección 5

Los campos identificadores y las variables de resultado posterior al corte quedan fuera de `X`; la etiqueta se conserva únicamente como `y`. Edad y comuna contienen faltantes y se tratarán en el pipeline. Los resultados de limpieza se obtuvieron exclusivamente del 80% de entrenamiento.

## 6. Variable objetivo

**Qué haremos:** mediremos la frecuencia de abandono y no abandono en train y mostraremos el desbalance de clases.

**Por qué es relevante:** la prevalencia define una referencia mínima para métricas como average precision y ayuda a interpretar precision, recall y la elección del umbral en el modelado.

In [6]:
target_summary = train_df[TARGET].value_counts().sort_index().rename(index={0: 'Sin abandono', 1: 'Abandono'}).to_frame('clientes')
target_summary['proporcion'] = target_summary.clientes / len(train_df)
display(target_summary)
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(target_summary.index, target_summary.clientes, color=[PALETTE[0], PALETTE[1]], label='Clientes train')
ax.bar_label(bars, labels=[f'{v:,.0f}\n{v / len(train_df):.1%}' for v in target_summary.clientes], padding=4, fontsize=9)
ax.set(title='Variable objetivo en train', xlabel='Resultado a seis meses', ylabel='Clientes')
ax.legend(frameon=False)
save_figure(fig, '01_distribucion_objetivo.png')

              clientes  proporcion
churn_6m                          
Sin abandono    112572    0.760622
Abandono         35428    0.239378
Gráfico guardado: outputs\graphs\01_distribucion_objetivo.png


### Conclusión de la sección 6

La tasa de abandono de train sirve como baseline para interpretar las métricas posteriores. Como la clase positiva es minoritaria, accuracy por sí sola no bastará; precision, recall, F1 y average precision serán más informativas. La proporción observada es sintética.

## 7. Variables numéricas: distribución y transformación

**Qué haremos:** calcularemos estadísticos descriptivos y asimetría, y graficaremos distribuciones numéricas disponibles en train.

**Por qué es relevante:** valores monetarios y conteos suelen tener colas largas y outliers. Reconocerlas permite elegir transformaciones y escaladores apropiados sin imponer una transformación que distorsione innecesariamente los datos.

In [7]:
non_predictor_cols = [
    ID, TARGET, 'fecha_registro', 'fecha_ultima_compra_pre_corte',
    'compras_periodo_objetivo', 'fecha_ultima_compra_al_cierre'
]
numeric_cols = [c for c in train_df.select_dtypes(include='number').columns if c not in non_predictor_cols]
numeric_profile = train_df[numeric_cols].describe(percentiles=[.25, .5, .75]).T
numeric_profile['asimetria'] = train_df[numeric_cols].skew(numeric_only=True)
display(numeric_profile.round(3))

plot_cols = numeric_cols.copy()
for index, col in enumerate(plot_cols, start=1):
    values = train_df[col].dropna()
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.histplot(values, bins=18, ax=ax, color=PALETTE[0], label=f'Train (n={len(values):,})')
    annotate_count_bars(ax, len(values))
    ax.set(title=f'Distribución de {col} | asimetría {train_df[col].skew():.2f}', xlabel=col, ylabel='Frecuencia')
    ax.legend(frameon=False)
    save_figure(fig, f'02_numerica_{index:02d}_{col}.png')

                                 count         mean  ...           max  asimetria
edad                          136230.0       48.933  ...  8.000000e+01      0.006
antiguedad_meses              148000.0       42.007  ...  7.200000e+01      0.001
cantidad_compras_12m          148000.0        7.995  ...  2.100000e+01      0.416
frecuencia_compra_mensual     148000.0        0.666  ...  1.750000e+00      0.416
monto_promedio_compra         148000.0    78542.946  ...  8.279196e+05      2.192
monto_total_12m               148000.0   628261.048  ...  7.451276e+06      2.479
proporcion_compras_promocion  148000.0        0.400  ...  1.000000e+00      0.304
devoluciones_12m              148000.0        0.402  ...  6.000000e+00      1.528
reclamos_12m                  148000.0        0.252  ...  5.000000e+00      1.992
interacciones_campanas_12m    148000.0        3.002  ...  1.300000e+01      0.579
recencia_dias                 148000.0       76.466  ...  3.650000e+02      1.362
cantidad_compras

### Conclusión de la sección 7

La asimetría y las colas largas de montos y conteos se deben valorar antes de escalar. La pipeline probará imputación por mediana y `RobustScaler` para variables numéricas, menos sensible a extremos que el escalado estándar; transformaciones logarítmicas solo se justificarán si mejoran validación y se aplican coherentemente a valores no negativos.

## 8. Variables categóricas y estrategia de codificación

**Qué haremos:** revisaremos cardinalidad, faltantes y frecuencia de niveles de las variables categóricas en train.

**Por qué es relevante:** la cardinalidad y el significado de las categorías determinan cómo representarlas. Una codificación inadecuada puede crear un orden artificial o aumentar excesivamente la dimensionalidad.

In [8]:
categorical_cols = [
    column for column in train_df.columns
    if pd.api.types.is_string_dtype(train_df[column].dtype)
    or isinstance(train_df[column].dtype, pd.CategoricalDtype)
]
categorical_profile = pd.DataFrame({
    'variable': categorical_cols,
    'cardinalidad': [train_df[c].nunique(dropna=True) for c in categorical_cols],
    'faltantes': [int(train_df[c].isna().sum()) for c in categorical_cols],
    'niveles_frecuentes': [train_df[c].value_counts(dropna=False).head(5).to_dict() for c in categorical_cols],
})
display(categorical_profile)
for index, col in enumerate(categorical_cols, start=1):
    counts = train_df[col].fillna('Sin dato').value_counts(dropna=False)
    fig, ax = plt.subplots(figsize=(max(7, len(counts) * 1.15), 4.8))
    bars = ax.bar(counts.index.astype(str), counts.values, color=PALETTE[:len(counts)], label=f'Train (n={len(train_df):,})')
    ax.bar_label(bars, labels=[f'{v:,.0f}\n{v / len(train_df):.1%}' for v in counts], padding=3, fontsize=8)
    ax.set(title=f'Frecuencia de {col}', xlabel=col, ylabel='Clientes')
    ax.tick_params(axis='x', rotation=25)
    ax.legend(frameon=False)
    save_figure(fig, f'03_categorica_{index:02d}_{col}.png')

              variable  ...                                 niveles_frecuentes
0               comuna  ...  {'Santiago': 31017, 'Puente Alto': 30802, 'Mai...
1               region  ...  {'Metropolitana': 96291, 'Valparaíso': 29478, ...
2         canal_compra  ...  {'Tienda física': 66432, 'Comercio electrónico...
3  categoria_principal  ...  {'Hogar': 53256, 'Tecnología': 50181, 'Electro...

[4 rows x 4 columns]
Gráfico guardado: outputs\graphs\03_categorica_01_comuna.png
Gráfico guardado: outputs\graphs\03_categorica_02_region.png
Gráfico guardado: outputs\graphs\03_categorica_03_canal_compra.png
Gráfico guardado: outputs\graphs\03_categorica_04_categoria_principal.png


### Conclusión de la sección 8

Canal y categoría de producto son nominales y se representarán con `OneHotEncoder(handle_unknown='ignore')`; no se asignará orden artificial. Comuna y región se mantienen para análisis descriptivo, pero quedan fuera del modelo inicial por cautela ante sesgos geográficos. No se usará target encoding por el riesgo de sobreajuste y fuga si no se calcula dentro de validación cruzada.

## 9. Relaciones entre variables categóricas y numéricas

**Qué haremos:** compararemos métricas de compra por canal y categoría, usando resúmenes y gráficos de distribución.

**Por qué es relevante:** los promedios agregados pueden ocultar diferencias entre segmentos. Estas comparaciones ayudan a formular hipótesis de negocio y a decidir qué variables conviene conservar para modelar.

In [9]:
display(train_df.groupby('canal_compra', observed=True)[['cantidad_compras_12m', 'monto_promedio_compra', 'recencia_dias']].agg(['mean', 'median']).round(2))
display(train_df.groupby('categoria_principal', observed=True)[['cantidad_compras_12m', 'monto_promedio_compra', 'proporcion_compras_promocion']].agg(['mean', 'median']).round(2))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=train_df, x='canal_compra', y='monto_promedio_compra', hue=TARGET, palette=PALETTE[:2], showfliers=False, ax=axes[0])
sns.boxplot(data=train_df, x='categoria_principal', y='monto_promedio_compra', hue=TARGET, palette=PALETTE[:2], showfliers=False, ax=axes[1])
for ax in axes:
    ax.tick_params(axis='x', rotation=20)
    ax.legend(title='Abandono (0/1)', labels=['No', 'Sí'], frameon=False)
axes[0].set_title('Ticket por canal y abandono')
axes[1].set_title('Ticket por categoría y abandono')
save_figure(fig, '04_ticket_por_segmento_y_abandono.png')

                     cantidad_compras_12m         ... recencia_dias       
                                     mean median  ...          mean median
canal_compra                                      ...                     
Comercio electrónico                 7.99    8.0  ...         76.20   64.0
Mixto                                8.00    8.0  ...         77.00   65.0
Tienda física                        7.99    8.0  ...         76.35   64.0

[3 rows x 6 columns]
                    cantidad_compras_12m  ... proporcion_compras_promocion
                                    mean  ...                       median
categoria_principal                       ...                             
Electrodomésticos                   8.01  ...                         0.38
Hogar                               8.00  ...                         0.38
Tecnología                          7.98  ...                         0.38

[3 rows x 6 columns]
Gráfico guardado: outputs\graphs\04_ticket_por_segmento_

### Conclusión de la sección 9

Las tablas y boxplots muestran cómo varían recencia, compras y ticket entre canales y categorías. Estas asociaciones sirven para orientar hipótesis; no implican que el canal o la categoría causen abandono ni prueban valor predictivo por sí solas.

## 10. Correlación y asociación con el objetivo

**Qué haremos:** visualizaremos correlaciones entre variables numéricas, asociación entre variables categóricas mediante Cramér V y asociación entre variables categóricas y numéricas mediante eta. También compararemos cada predictor con el objetivo.

**Por qué es relevante:** la redundancia entre predictores puede complicar la interpretación y no aportar información adicional. La correlación es solo una señal de asociación; no sustituye validación predictiva y no detecta todas las relaciones no lineales.

In [10]:
corr_cols = [c for c in numeric_cols if c != ID]
corr = train_df[corr_cols].corr(method='spearman')
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, cmap='vlag', center=0, annot=True, fmt='.2f', annot_kws={'size': 7}, cbar_kws={'label': 'Correlación Spearman'}, ax=ax)
ax.set_title('Correlación Spearman entre predictores numéricos de train')
save_figure(fig, '05_correlacion_numerica.png')

target_corr = train_df[corr_cols].corrwith(train_df[TARGET]).sort_values(key=lambda s: s.abs(), ascending=False)
display(target_corr.rename('correlacion_con_churn').to_frame().round(3))
for col in categorical_cols:
    rates = train_df.groupby(col, dropna=False, observed=True)[TARGET].agg(['mean', 'size']).sort_values('mean', ascending=False)
    print(f'Tasa de abandono por {col}')
    display(rates.rename(columns={'mean': 'tasa_abandono', 'size': 'clientes'}).round(3))

from scipy.stats import chi2_contingency

def cramers_v(left, right):
    table = pd.crosstab(left.fillna('Sin dato'), right.fillna('Sin dato'))
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    rows, cols = table.shape
    if n <= 1 or min(rows, cols) <= 1:
        return 0.0
    phi2 = chi2 / n
    correction = ((cols - 1) * (rows - 1)) / (n - 1)
    phi2_corrected = max(0, phi2 - correction)
    rows_corrected = rows - ((rows - 1) ** 2) / (n - 1)
    cols_corrected = cols - ((cols - 1) ** 2) / (n - 1)
    denominator = min(cols_corrected - 1, rows_corrected - 1)
    return np.sqrt(phi2_corrected / denominator) if denominator > 0 else 0.0

category_association = pd.DataFrame(index=categorical_cols, columns=categorical_cols, dtype=float)
for left in categorical_cols:
    for right in categorical_cols:
        category_association.loc[left, right] = 1.0 if left == right else cramers_v(train_df[left], train_df[right])
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(category_association, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar_kws={'label': 'Cramér V'}, ax=ax)
ax.set_title('Asociación entre variables categóricas (train)')
save_figure(fig, '06_asociacion_categorica.png')

mixed_numeric = [*numeric_cols, TARGET]
eta = pd.DataFrame(index=categorical_cols, columns=mixed_numeric, dtype=float)
for category in categorical_cols:
    groups = train_df.groupby(category, dropna=False, observed=True)
    for number in mixed_numeric:
        values = train_df[number]
        mean = values.mean()
        grouped = groups[number].agg(['mean', 'count'])
        between = (((grouped['mean'] - mean) ** 2) * grouped['count']).sum()
        total = ((values - mean) ** 2).sum()
        eta.loc[category, number] = np.sqrt(between / total) if total > 0 else 0.0
fig, ax = plt.subplots(figsize=(15, 4))
sns.heatmap(eta.astype(float), annot=True, fmt='.2f', cmap='YlOrBr', vmin=0, vmax=1, cbar_kws={'label': 'Eta'}, ax=ax)
ax.set_title('Asociación entre categorías y variables numéricas / objetivo (train)')
save_figure(fig, '07_asociacion_categorica_numerica.png')

Gráfico guardado: outputs\graphs\05_correlacion_numerica.png
                              correlacion_con_churn
recencia_dias                                 0.211
reclamos_12m                                  0.111
interacciones_campanas_12m                   -0.083
cantidad_compras_12m                         -0.083
frecuencia_compra_mensual                    -0.083
proporcion_compras_promocion                  0.038
cantidad_compras_36m                         -0.034
monto_total_12m                              -0.033
devoluciones_12m                              0.023
monto_total_36m                              -0.014
antiguedad_meses                             -0.004
edad                                          0.001
monto_promedio_compra                        -0.001
Tasa de abandono por comuna
              tasa_abandono  clientes
comuna                               
NaN                   0.245      5900
Los Ángeles           0.243      7190
Quilpué               0.242    

### Conclusión de la sección 10

Las matrices ayudan a localizar redundancia numérica y asociaciones entre segmentos; se excluyeron ID, fechas crudas y variables de la ventana futura para no introducir fuga en la exploración predictiva. Frecuencia mensual, compras anuales, monto total y ticket merecen revisión por redundancia. Cramér V y eta describen asociaciones, no causalidad; la selección final se contrastará en train y la generalización solo en el holdout al final.

## 11. Distribuciones numéricas según la variable objetivo

**Qué haremos:** compararemos distribuciones de predictores entre abandono y no abandono mediante boxplots y medianas.

**Por qué es relevante:** la separación visual entre distribuciones puede sugerir señales predictivas y revelar outliers. El solapamiento importa: diferencias de medias no bastan para afirmar que una variable clasifica bien.

In [11]:
box_cols = [c for c in numeric_cols if c not in [ID] and train_df[c].nunique(dropna=True) > 2]
summary_by_target = train_df.groupby(TARGET)[box_cols].median().T
summary_by_target.columns = ['mediana_sin_abandono', 'mediana_abandono']
summary_by_target['diferencia_absoluta'] = (summary_by_target.iloc[:, 1] - summary_by_target.iloc[:, 0]).abs()
display(summary_by_target.sort_values('diferencia_absoluta', ascending=False).round(2))
train_plot = train_df.sample(n=min(15000, len(train_df)), random_state=42)
for index, col in enumerate(box_cols, start=1):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(data=train_plot, x=TARGET, y=col, hue=TARGET, palette=PALETTE[:2], showfliers=False, ax=ax)
    ax.set(title=f'{col} por resultado (muestra n={len(train_plot):,})', xlabel='Abandono a seis meses', ylabel=col)
    ax.set_xticks([0, 1], labels=['Sin abandono', 'Abandono'])
    ax.legend(title='Clase', labels=['Sin abandono', 'Abandono'], frameon=False)
    save_figure(fig, f'08_boxplot_{index:02d}_{col}.png')

                              mediana_sin_abandono  ...  diferencia_absoluta
monto_total_36m                         1331224.50  ...             44262.00
monto_total_12m                          509625.01  ...             34043.48
monto_promedio_compra                     65527.01  ...               420.51
recencia_dias                                60.00  ...                23.00
cantidad_compras_12m                          8.00  ...                 1.00
cantidad_compras_36m                         22.00  ...                 1.00
frecuencia_compra_mensual                     0.67  ...                 0.08
proporcion_compras_promocion                  0.38  ...                 0.03
edad                                         49.00  ...                 0.00
antiguedad_meses                             42.00  ...                 0.00
devoluciones_12m                              0.00  ...                 0.00
reclamos_12m                                  0.00  ...                 0.00

### Conclusión de la sección 11

Los boxplots permiten comparar medianas y dispersión, pero no garantizan predicción cuando las cajas se solapan. Recencia, compras, reclamos y devoluciones merecen revisión como señales plausibles del patrón simulado; su contribución real se determinará con modelos y validación, no por inspección visual.

## 12. Tendencias y gráficos de distribución

**Qué haremos:** compararemos tendencias descriptivas de recencia y compras por resultado, y la composición por región.

**Por qué es relevante:** gráficos resumidos facilitan comunicar patrones de compra y abandono a negocio. Se limitan a train para mantener el holdout sin exposición.

In [12]:
train_df['segmento_recencia_eda'] = pd.cut(train_df['recencia_dias'], bins=[0, 30, 90, 180, 366], labels=['0-30 días', '31-90 días', '91-180 días', '181+ días'], include_lowest=True)
trend_specs = [('segmento_recencia_eda', '09_tendencia_recencia.png', 'Clientes por segmento de recencia'), ('cantidad_compras_12m', '10_tendencia_compras.png', 'Clientes por compras previas'), ('region', '11_tendencia_region.png', 'Clientes por región')]
for column, filename, title in trend_specs:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.countplot(data=train_df, x=column, hue=TARGET, palette=PALETTE[:2], ax=ax)
    for patch in ax.patches:
        value = patch.get_height()
        if value > 0:
            ax.annotate(f'{value:,.0f}\n{value / len(train_df):.1%}', (patch.get_x() + patch.get_width() / 2, value), ha='center', va='bottom', fontsize=7)
    ax.set(title=title, xlabel=column, ylabel='Clientes')
    ax.legend(title='Abandono (0/1)', labels=['No', 'Sí'], frameon=False)
    ax.tick_params(axis='x', rotation=20)
    save_figure(fig, filename)

Gráfico guardado: outputs\graphs\09_tendencia_recencia.png
Gráfico guardado: outputs\graphs\10_tendencia_compras.png
Gráfico guardado: outputs\graphs\11_tendencia_region.png


### Conclusión de la sección 12

Las figuras comunican diferencias de composición y comportamiento dentro de train y sirven para orientar la conversación de negocio. La distribución regional también recuerda que el volumen de cada grupo no es igual; por eso las métricas segmentadas del modelo deberán interpretarse con cautela.

## 13. Conclusión del análisis exploratorio

**Qué haremos:** sintetizaremos los hallazgos del EDA antes de pasar a ingeniería de características y ETL.

**Por qué es relevante:** un cierre explícito convierte las visualizaciones en decisiones documentadas y permite rastrear cómo las conclusiones exploratorias se traducen en preparación de datos.

In [13]:
print(f'Train disponible para el EDA: {len(train_df):,} filas.')
print(f'Tasa de abandono simulada en train: {train_df[TARGET].mean():.2%}.')
print('Revisar asimetría y valores extremos antes de escoger escalado o transformaciones.')
print('Excluir ID, fechas crudas y variables que describen la ventana futura.')
print('Mantener el holdout cerrado hasta la evaluación final del modelo.')

Train disponible para el EDA: 148,000 filas.
Tasa de abandono simulada en train: 23.94%.
Revisar asimetría y valores extremos antes de escoger escalado o transformaciones.
Excluir ID, fechas crudas y variables que describen la ventana futura.
Mantener el holdout cerrado hasta la evaluación final del modelo.


### Conclusión de la sección 13

El EDA establece una línea base de calidad, prevalencia y asociaciones en train. Las decisiones preliminares son: imputar dentro de una pipeline, escalar numéricos de cola larga con métodos robustos, codificar categorías nominales con one-hot, excluir geografía del modelo inicial y revisar redundancia. Ninguna asociación observada demuestra causalidad ni rendimiento en clientes reales.

## 14. Ingeniería de características para el modelo y el dashboard

**Qué haremos:** derivaremos indicadores basados exclusivamente en actividad anterior al corte, útiles para enriquecer el análisis y proponer KPIs para el futuro dashboard.

**Por qué es relevante:** indicadores interpretables acercan los datos a preguntas de negocio. Las nuevas variables se calculan después del split y sobre train; antes de incorporarlas al modelo se deben validar dentro de la pipeline.

In [14]:
train_df['tasa_devoluciones'] = train_df['devoluciones_12m'] / train_df['cantidad_compras_12m'].clip(lower=1)
train_df['gasto_mensual_estimado'] = train_df['monto_total_12m'] / 12
train_df['segmento_recencia'] = pd.cut(
    train_df['recencia_dias'], bins=[0, 30, 90, 180, 366],
    labels=['0-30 días', '31-90 días', '91-180 días', '181+ días'], include_lowest=True
)
display(train_df[['tasa_devoluciones', 'gasto_mensual_estimado', 'segmento_recencia']].describe(include='all').T)

                           count unique  ...           75%            max
tasa_devoluciones       148000.0    NaN  ...           0.1           0.75
gasto_mensual_estimado  148000.0    NaN  ...  66060.614583  620939.666667
segmento_recencia         148000      4  ...           NaN            NaN

[3 rows x 11 columns]


### Conclusión de la sección 14

`tasa_devoluciones` resume devoluciones relativas a compras; `gasto_mensual_estimado` y `segmento_recencia` son candidatos de KPI. Solo `tasa_devoluciones` se considera candidata para el modelo inicial; los otros indicadores se exportan para exploración/dashboard y no se incorporan automáticamente a la predicción.

## 15. Diagnóstico de varianza, correlación y selección de variables

**Qué haremos:** buscaremos columnas constantes y pares numéricos altamente correlacionados, y compararemos selección supervisada con `SelectKBest` y `SelectFromModel`.

**Por qué es relevante:** reducir variables redundantes puede simplificar el entrenamiento. Como `churn_6m` es binaria, una regresión lineal ordinaria no es la herramienta adecuada para selección; usaremos regresión logística regularizada y criterios de información mutua, siempre ajustados con train. Estos diagnósticos son candidatos, no una selección final automática.

In [15]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, SelectFromModel
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression

MODEL_FEATURES = [
    'edad', 'canal_compra', 'antiguedad_meses', 'cantidad_compras_12m',
    'monto_promedio_compra', 'categoria_principal', 'proporcion_compras_promocion',
    'devoluciones_12m', 'reclamos_12m', 'interacciones_campanas_12m',
    'recencia_dias', 'tasa_devoluciones'
]
X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET].astype(int).copy()
num_features = X_train.select_dtypes(include='number').columns.tolist()
cat_features = [c for c in MODEL_FEATURES if c not in num_features]

imputed_numeric = X_train[num_features].fillna(X_train[num_features].median())
variance = VarianceThreshold(threshold=0.0).fit(imputed_numeric)
zero_variance = [c for c, keep in zip(num_features, variance.get_support()) if not keep]
print('Variables numéricas de varianza cero:', zero_variance or 'ninguna')

numeric_corr = imputed_numeric.corr().abs()
high_corr_pairs = [(a, b, numeric_corr.loc[a, b]) for i, a in enumerate(num_features) for b in num_features[i+1:] if numeric_corr.loc[a, b] >= .95]
print('Pares numéricos con |correlación Pearson| >= 0.95:')
display(pd.DataFrame(high_corr_pairs, columns=['variable_1', 'variable_2', 'correlacion']) if high_corr_pairs else pd.DataFrame(columns=['variable_1', 'variable_2', 'correlacion']))

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median', add_indicator=True)), ('scale', RobustScaler())]), num_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), cat_features),
])
X_encoded = preprocessor.fit_transform(X_train)
encoded_names = preprocessor.get_feature_names_out()
k = min(10, X_encoded.shape[1])
discrete_mask = np.array([name.startswith('cat__') for name in encoded_names])
kbest = SelectKBest(score_func=lambda X, y: mutual_info_classif(X, y, discrete_features=discrete_mask, random_state=42), k=k).fit(X_encoded, y_train)
mi_results = pd.DataFrame({'variable_transformada': encoded_names, 'informacion_mutua': kbest.scores_, 'seleccionada': kbest.get_support()})
display(mi_results.sort_values('informacion_mutua', ascending=False).head(15))

selector_model = SelectFromModel(LogisticRegression(penalty='l1', solver='liblinear', class_weight='balanced', max_iter=500, random_state=42), threshold='median')
selector_model.fit(X_encoded, y_train)
print('SelectFromModel seleccionó:', list(encoded_names[selector_model.get_support()]))

Variables numéricas de varianza cero: ninguna
Pares numéricos con |correlación Pearson| >= 0.95:
Empty DataFrame
Columns: [variable_1, variable_2, correlacion]
Index: []
                         variable_transformada  informacion_mutua  seleccionada
8                           num__recencia_dias       2.069518e-02          True
6                            num__reclamos_12m       5.844060e-03          True
7              num__interacciones_campanas_12m       5.075324e-03          True
5                        num__devoluciones_12m       2.966104e-03          True
0                                    num__edad       2.749818e-03          True
2                    num__cantidad_compras_12m       2.413907e-03          True
4            num__proporcion_compras_promocion       2.391847e-03          True
9                       num__tasa_devoluciones       1.708715e-03          True
10                  num__missingindicator_edad       1.030415e-03          True
12                     cat__ca

C:\Users\shein\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\shein\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


### Conclusión de la sección 15

Los diagnósticos señalan variables constantes o redundantes y producen listas de candidatos mediante dos métodos supervisados. La imputación, escalado y codificación de esta celda son exploratorios de train; para estimar generalización, la pipeline debe volver a ajustarlos dentro de cada partición de entrenamiento. Las selecciones no se aplican automáticamente porque deben contrastarse en validación y con criterios de negocio.

## 16. ETL y exportaciones para modelado y dashboard

**Qué haremos:** construiremos X/y de entrenamiento sin identificadores ni fugas, guardaremos el train limpio y un extracto con nombres de negocio y KPIs, y mantendremos el holdout separado.

**Por qué es relevante:** los artefactos estables permiten conectar el EDA con la pipeline de entrenamiento y preparar una fuente inicial para Power BI. Las transformaciones aprendidas por el modelo deben permanecer dentro de su pipeline para ajustarse solo con los datos de entrenamiento de cada validación.

In [16]:
X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df[TARGET].astype(int).copy()
train_source = train_df[[ID, 'region', *MODEL_FEATURES, TARGET]].copy()
train_source_path = TRAIN_SOURCE_PATH
train_source.to_csv(train_source_path, index=False)
train_clean = train_source.copy()
for feature in num_features:
    train_clean[feature] = train_clean[feature].fillna(X_train[feature].median())
for feature in cat_features:
    train_clean[feature] = train_clean[feature].fillna(X_train[feature].mode(dropna=True).iloc[0])
train_clean_path = TRAIN_CLEAN_PATH
train_clean.to_csv(train_clean_path, index=False)

dashboard = pd.DataFrame({
    'customer_id': train_df[ID],
    'region': train_df['region'],
    'purchase_channel': train_df['canal_compra'],
    'main_category': train_df['categoria_principal'],
    'purchases_12m': train_df['cantidad_compras_12m'],
    'average_purchase_value': train_df['monto_promedio_compra'],
    'estimated_monthly_spend': train_df['gasto_mensual_estimado'],
    'recency_days': train_df['recencia_dias'],
    'recency_segment': train_df['segmento_recencia'].astype('string'),
    'return_rate': train_df['tasa_devoluciones'],
    'complaints_12m': train_df['reclamos_12m'],
    'promotion_purchase_share': train_df['proporcion_compras_promocion'],
    'observed_churn_6m': train_df[TARGET],
})
dashboard_path = DASHBOARD_TRAIN_PATH
dashboard.to_csv(dashboard_path, index=False)
print(f'X_train: {X_train.shape}; y_train: {y_train.shape}')
print(f'Train limpio e imputado con estadísticos de train: {train_clean_path.relative_to(ROOT)}')
print(f'Fuente train con faltantes para validación sin fuga: {train_source_path.relative_to(ROOT)}')
print(f'Datos para dashboard (solo train): {dashboard_path.relative_to(ROOT)}')
print(f'Test holdout intacto: {TEST_RAW_PATH.relative_to(ROOT)}')
display(dashboard.head())

X_train: (148000, 12); y_train: (148000,)
Train limpio e imputado con estadísticos de train: data\processed\train\train_clean.csv
Fuente train con faltantes para validación sin fuga: data\processed\train\train_source.csv
Datos para dashboard (solo train): data\processed\train\dashboard_train.csv
Test holdout intacto: data\raw\test\test.csv
   customer_id         region  ... promotion_purchase_share observed_churn_6m
0       129975     Valparaíso  ...                 0.200000                 0
1        24230  Metropolitana  ...                 0.818182                 0
2        66327  Metropolitana  ...                 0.000000                 0
3        69942     Valparaíso  ...                 0.125000                 0
4       119494         Biobío  ...                 0.571429                 0

[5 rows x 13 columns]


### Conclusión de la sección 16

Se exportaron `data/processed/train/train_clean.csv` (train imputado con estadísticos de train), `data/processed/train/train_source.csv` (valores originales con faltantes para validación sin fuga) y `data/processed/train/dashboard_train.csv`; todos corresponden exclusivamente a train. `data/raw/test/test.csv` queda aislado para la evaluación final. El modelado consume los splits raw y ajusta imputación, escalado y encoding dentro del pipeline. El CSV de dashboard es una base inicial sin predicciones.